## **Install the necessary libraries**

In [ ]:
!pip install ultralytics

## **Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Import libraries**

In [ ]:
from google.colab import drive
from ultralytics import YOLO
import os, shutil, yaml, random

import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## **Define the Data path**

*   Update dataset paths (DATA_1, DATA_2, DATA_3)
Change these to the exact folders in your Google Drive where your YOLO datasets are stored.

*   Update merged output path (DATA_MERGED)
This is where all datasets will be combined. Use any folder name you prefer.

*  Update model path (OLD_MODEL)
Point this to your existing trained .pt model file.
If you don’t have one, you can skip this or train from scratch.

*  Do not change the validation logic
The code checks whether DATA_3 exists to avoid training with missing data.

*  Define the path correctly so that error does not occur

In [ ]:
# Your dataset paths
DATA_1 = '/content/drive/MyDrive/Data_1_'
DATA_2 = '/content/drive/MyDrive/Data_2_'
DATA_3 = '/content/drive/MyDrive/Data_3_'
DATA_MERGED = '/content/drive/MyDrive/Data_merged_all'

# Your model paths
OLD_MODEL = '/content/drive/MyDrive/models/v2.0_multiclass.pt' # This is optional if there specify for better comparison between two models
MODELS_DIR = '/content/drive/MyDrive/models' # This is optional

print("="*70)
print("📁 PATHS CONFIGURATION")
print("="*70)
print(f"Data_1: {DATA_1}")
print(f"Data_2: {DATA_2}")
print(f"Data_3: {DATA_3} (NEW)")
print(f"Output: {DATA_MERGED}")
print(f"Current model: {OLD_MODEL}")
print("="*70)

# Verify Data_3 exists
if not os.path.exists(DATA_3):
    print("\n❌ DATA_3 NOT FOUND!")
    print("\n📝 STEPS TO CREATE DATA_3:")
    print("1. Upload new images to Google Drive")
    print("2. Annotate in Roboflow")
    print("3. Export as YOLOv8")
    print("4. Extract to:", DATA_3)
    print("\n⏸️ STOP HERE - Create Data_3 first!")
else:
    print("\n✅ Data_3 found!")

## **Validating/Checking the dataset**

In [ ]:
def analyze_dataset(dataset_path, name):
    """Analyze a single dataset"""

    if not os.path.exists(dataset_path):
        print(f"❌ {name} not found at: {dataset_path}")
        return None

    yaml_path = f"{dataset_path}/data.yaml"

    if not os.path.exists(yaml_path):
        print(f"❌ data.yaml not found in {name}")
        return None

    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # Count images
    total_images = 0
    for split in ['train', 'valid', 'test']:
        img_dir = f"{dataset_path}/{split}/images"
        if os.path.exists(img_dir):
            count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
            total_images += count

    return {
        'name': name,
        'path': dataset_path,
        'classes': config['names'],
        'num_classes': config['nc'],
        'total_images': total_images
    }

# Analyze all datasets
print("="*70)
print("📊 DATASET ANALYSIS")
print("="*70)

datasets_info = []

for path, name in [(DATA_1, 'Data_1'), (DATA_2, 'Data_2'), (DATA_3, 'Data_3')]:
    info = analyze_dataset(path, name)
    if info:
        datasets_info.append(info)
        print(f"\n{name}:")
        print(f"   Images: {info['total_images']}")
        print(f"   Classes: {info['num_classes']}")
        print(f"   Names: {info['classes']}")

# Summary
if len(datasets_info) == 3:
    all_classes = []
    for ds in datasets_info:
        all_classes.extend(ds['classes'])

    unique_classes = list(set(all_classes))
    total_images = sum([ds['total_images'] for ds in datasets_info])

    print("\n" + "="*70)
    print("📈 COMBINED SUMMARY")
    print("="*70)
    print(f"Total datasets: {len(datasets_info)}")
    print(f"Total images: {total_images}")
    print(f"Unique classes: {len(unique_classes)}")
    print(f"All classes: {sorted(unique_classes)}")
    print("="*70)
else:
    print("\n⚠️ Not all datasets are available. Please check paths.")

## **Combining all the dataset as One**

In [ ]:
def merge_multiple_datasets(dataset_paths, output_path):
    """
    Merge multiple datasets with intelligent class remapping

    Args:
        dataset_paths: List of dataset paths
        output_path: Where to save merged dataset
    """

    print("="*70)
    print("🔄 MERGING MULTIPLE DATASETS")
    print("="*70)

    # Clean output directory
    if os.path.exists(output_path):
        print("🗑️ Removing old merged dataset...")
        shutil.rmtree(output_path)

    # Step 1: Collect all unique classes
    all_classes = []
    dataset_configs = []

    for dataset_path in dataset_paths:
        with open(f"{dataset_path}/data.yaml", 'r') as f:
            config = yaml.safe_load(f)

        dataset_configs.append({
            'path': dataset_path,
            'config': config,
            'classes': config['names']
        })

        # Add new classes
        for cls in config['names']:
            if cls not in all_classes:
                all_classes.append(cls)

    print(f"\n📊 Class Consolidation:")
    for i, ds_config in enumerate(dataset_configs, 1):
        print(f"   Dataset {i}: {len(ds_config['classes'])} classes - {ds_config['classes']}")

    print(f"\n✅ Final merged classes: {len(all_classes)}")
    print(f"   {all_classes}")

    # Step 2: Create class mappings for each dataset
    class_mappings = []
    for ds_config in dataset_configs:
        mapping = {}
        for old_id, class_name in enumerate(ds_config['classes']):
            new_id = all_classes.index(class_name)
            mapping[old_id] = new_id
        class_mappings.append(mapping)

    # Step 3: Collect all images with their mappings
    all_images = []

    for dataset_idx, ds_config in enumerate(dataset_configs):
        dataset_path = ds_config['path']
        dataset_name = os.path.basename(dataset_path)
        count = 0

        for split in ['train', 'valid', 'test']:
            img_dir = f"{dataset_path}/{split}/images"

            if os.path.exists(img_dir):
                for img in os.listdir(img_dir):
                    if img.endswith(('.jpg', '.jpeg', '.png')):
                        all_images.append({
                            'source_path': dataset_path,
                            'source_split': split,
                            'image_name': img,
                            'class_mapping': class_mappings[dataset_idx],
                            'source_classes': ds_config['classes']
                        })
                        count += 1

        print(f"   ✅ {dataset_name}: {count} images collected")

    print(f"\n📊 Total images collected: {len(all_images)}")

    # Step 4: Shuffle and re-split
    print("\n🔀 Shuffling and re-splitting...")
    random.shuffle(all_images)

    train_size = int(0.7 * len(all_images))
    valid_size = int(0.2 * len(all_images))

    splits = {
        'train': all_images[:train_size],
        'valid': all_images[train_size:train_size + valid_size],
        'test': all_images[train_size + valid_size:]
    }

    print(f"   Train: {len(splits['train'])} ({len(splits['train'])/len(all_images)*100:.1f}%)")
    print(f"   Valid: {len(splits['valid'])} ({len(splits['valid'])/len(all_images)*100:.1f}%)")
    print(f"   Test: {len(splits['test'])} ({len(splits['test'])/len(all_images)*100:.1f}%)")

    # Step 5: Create output structure
    for split in ['train', 'valid', 'test']:
        os.makedirs(f"{output_path}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_path}/{split}/labels", exist_ok=True)

    # Step 6: Copy images and remap labels
    print("\n📋 Copying files and remapping class IDs...")

    for split_name, image_list in splits.items():
        copied = 0

        for img_info in image_list:
            # Source paths
            img_src = f"{img_info['source_path']}/{img_info['source_split']}/images/{img_info['image_name']}"
            lbl_name = img_info['image_name'].replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
            lbl_src = f"{img_info['source_path']}/{img_info['source_split']}/labels/{lbl_name}"

            # Destination paths
            img_dst = f"{output_path}/{split_name}/images/{img_info['image_name']}"
            lbl_dst = f"{output_path}/{split_name}/labels/{lbl_name}"

            # Copy image
            if os.path.exists(img_src):
                shutil.copy(img_src, img_dst)
            else:
                continue

            # Remap and copy label
            if os.path.exists(lbl_src):
                with open(lbl_src, 'r') as f:
                    lines = f.readlines()

                with open(lbl_dst, 'w') as f:
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            old_class_id = int(parts[0])

                            # Remap class ID
                            new_class_id = img_info['class_mapping'][old_class_id]

                            # Write with new class ID
                            f.write(f"{new_class_id} {' '.join(parts[1:])}\n")

                copied += 1

        print(f"   ✅ {split_name}: {copied} files copied")

    # Step 7: Create data.yaml
    print("\n📝 Creating data.yaml...")

    merged_config = {
        'path': output_path,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(all_classes),
        'names': all_classes
    }

    yaml_path = f"{output_path}/data.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump(merged_config, f, default_flow_style=False)

    print("   ✅ data.yaml created")

    # Summary
    print("\n" + "="*70)
    print("✅ MERGE COMPLETE!")
    print("="*70)
    print(f"📁 Location: {output_path}")
    print(f"📊 Total classes: {len(all_classes)}")
    print(f"📊 Total images: {len(all_images)}")
    print(f"   Train: {len(splits['train'])}")
    print(f"   Valid: {len(splits['valid'])}")
    print(f"   Test: {len(splits['test'])}")
    print("="*70)

    return yaml_path, all_classes, len(all_images)

# Execute merge
merged_yaml, final_classes, total_images = merge_multiple_datasets(
    dataset_paths=[DATA_1, DATA_2, DATA_3],
    output_path=DATA_MERGED
)

print(f"\n📋 Final class list: {final_classes}")

## **Verfiying the combined dataset**

In [ ]:
print("="*70)
print("🔍 VERIFYING MERGED DATASET")
print("="*70)

# Check structure
for split in ['train', 'valid', 'test']:
    img_dir = f"{DATA_MERGED}/{split}/images"
    lbl_dir = f"{DATA_MERGED}/{split}/labels"

    if os.path.exists(img_dir):
        num_images = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        num_labels = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
        print(f"✅ {split}: {num_images} images, {num_labels} labels")

        if num_images != num_labels:
            print(f"   ⚠️ Warning: Mismatch between images and labels")
    else:
        print(f"❌ {split} folder not found")

# Check data.yaml
with open(f"{DATA_MERGED}/data.yaml", 'r') as f:
    config = yaml.safe_load(f)

print(f"\n✅ data.yaml:")
print(f"   Classes: {config['nc']}")
print(f"   Names: {config['names']}")

# Sample a few labels to verify class IDs are correct
print(f"\n🔍 Sampling labels to verify class remapping...")

sample_split = 'train'
label_dir = f"{DATA_MERGED}/{sample_split}/labels"
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')][:5]

class_id_counts = Counter()

for lbl_file in label_files:
    with open(f"{label_dir}/{lbl_file}", 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                class_id_counts[class_id] += 1

print(f"   Sample class IDs found: {dict(class_id_counts)}")
print(f"   Valid range: 0 to {config['nc']-1}")

if max(class_id_counts.keys()) >= config['nc']:
    print("   ❌ ERROR: Invalid class IDs found!")
else:
    print("   ✅ All class IDs are valid!")

print("\n" + "="*70)
print("✅ VERIFICATION COMPLETE!")
print("="*70)

## **Training the model**

In [ ]:
model = YOLO('yolov8n.pt')

print("="*70)
print("🔄 RETRAINING MODEL WITH ALL DATA")
print("="*70)
print(f"📊 Total classes: {len(final_classes)}")
print(f"📊 Total images: {total_images}")
print(f"📦 Starting from: YOLOv8 pretrained (fresh start)")
print("="*70)

results = model.train(
    data=merged_yaml,
   # Training parameters
    epochs=150,
    imgsz=640,
    batch=16,

    # Model saving
    name='model_v3_all_data', # can give any output name
    save=True,
    save_period=10,

    # Early stopping
    patience=20,             # Stop if no improvement for N epochs

    # Optimization
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    # Augmentation
    hsv_h=0.015,             # HSV-Hue augmentation
    hsv_s=0.7,               # HSV-Saturation
    hsv_v=0.4,               # HSV-Value
    degrees=10,              # Rotation (+/- deg)
    translate=0.1,           # Translation (+/- fraction)
    scale=0.5,               # Scale (+/- gain)
    shear=0.0,               # Shear (+/- deg)
    perspective=0.0,         # Perspective (+/- fraction)
    flipud=0.0,              # Flip up-down probability
    fliplr=0.5,              # Flip left-right probability
    mosaic=1.0,              # Mosaic augmentation probability
    mixup=0.1,               # Mixup augmentation probability
    copy_paste=0.0,          # Copy-paste augmentation probability

    # Visualization
    plots=True,              # Save training plots
    verbose=True,            # Verbose output

    # Device
    device=0,

    # Advanced
    exist_ok=True,           # Overwrite existing experiment
    pretrained=True,         # Use pretrained weights
    resume=False,            # Resume from last checkpoint
)

print("\n" + "="*50)
print("✅ TRAINING COMPLETE!")
print("="*50)

## **Evaluation Metrics**

In [ ]:
# Load trained model
trained_model = YOLO('runs/detect/model_v3_all_data/weights/best.pt')

# Validate
print("🔍 Evaluating trained model...")
metrics = trained_model.val(data=merged_yaml)

print("\n" + "="*70)
print("📊 MODEL EVALUATION RESULTS")
print("="*70)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

# Calculate F1
if metrics.box.mp > 0 and metrics.box.mr > 0:
    f1 = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr)
    print(f"F1-Score: {f1:.4f}")

print("="*70)

# Per-class metrics
print("\n📊 Per-Class Performance:")
print("-" * 50)

if hasattr(metrics.box, 'maps'):
    for i, class_name in enumerate(final_classes):
        if i < len(metrics.box.maps):
            class_map = metrics.box.maps[i]
            print(f"   {class_name}: {class_map:.3f}")

## **Saving the output models**

In [ ]:
import json
from datetime import datetime

# Version info
VERSION = 'v3.0'
MODELS_DIR = '/content/drive/MyDrive/models2'

# Create version directory
version_dir = f'{MODELS_DIR}/{VERSION}'
os.makedirs(version_dir, exist_ok=True)

print("="*70)
print("💾 SAVING MODEL")
print("="*70)

# Copy best model
shutil.copy(
    'runs/detect/model_v3_all_data/weights/best.pt',
    f'{version_dir}/best.pt'
)
print(f"✅ Saved: {version_dir}/best.pt")

# Copy last model (for resume training)
shutil.copy(
    'runs/detect/model_v3_all_data/weights/last.pt',
    f'{version_dir}/last.pt'
)
print(f"✅ Saved: {version_dir}/last.pt")

# Copy training results
shutil.copytree(
    'runs/detect/model_v3_all_data',
    f'{version_dir}/training_results',
    dirs_exist_ok=True
)
print(f"✅ Saved: {version_dir}/training_results/")

# Update latest model
shutil.copy(
    'runs/detect/model_v3_all_data/weights/best.pt',
    f'{MODELS_DIR}/latest.pt'
)
print(f"✅ Updated: {MODELS_DIR}/latest.pt")

# Create metadata
metadata = {
    'version': VERSION,
    'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'datasets_used': ['Data_1', 'Data_2', 'Data_3'],
    'total_images': total_images,
    'total_classes': len(final_classes),
    'class_names': final_classes,
    'epochs': 100,
    'mAP@0.5': float(metrics.box.map50),
    'mAP@0.5:0.95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr)
}

with open(f'{version_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved: {version_dir}/metadata.json")

print("\n" + "="*70)
print("✅ MODEL SAVED SUCCESSFULLY!")
print("="*70)
print(f"📁 Version: {VERSION}")
print(f"📁 Location: {version_dir}")
print(f"📊 Classes: {len(final_classes)}")
print(f"📊 Images: {total_images}")
print(f"📈 mAP@0.5: {metadata['mAP@0.5']:.4f}")
print("="*70)

## **Perdiciting on new images**

In [ ]:
# Load new model
model_v3 = YOLO(f'{MODELS_DIR}/{VERSION}/best.pt')

print("="*70)
print("🎯 TESTING NEW MODEL")
print("="*70)

# Option 1: Test on uploaded image
from google.colab import files

print("\n📤 Upload a test image:")
uploaded = files.upload()

if uploaded:
    test_image = list(uploaded.keys())[0]

    # Predict
    results = model_v3.predict(
        source=test_image,
        conf=0.5,
        save=True,
        project='test_results',
        name='v3_predictions'
    )

    print("\n🎯 PREDICTIONS:")
    print("="*60)

    if len(results[0].boxes) > 0:
        for box in results[0].boxes:
            class_name = model_v3.names[int(box.cls[0])]
            confidence = float(box.conf[0])
            print(f"  ✅ {class_name}: {confidence:.1%}")
    else:
        print("  ⚠️ No detections")

    print("="*60)

    # Display result
    from IPython.display import Image, display
    result_path = f'test_results/v3_predictions/{test_image}'
    if os.path.exists(result_path):
        display(Image(result_path))
        print(f"\n📁 Result saved: {result_path}")
else:
    print("⚠️ No image uploaded")

# Option 2: Test on validation set sample
print("\n📊 Testing on validation set samples...")

val_images_dir = f'{DATA_MERGED}/valid/images'
val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.png'))][:3]

for img_name in val_images:
    img_path = f'{val_images_dir}/{img_name}'
    results = model_v3.predict(img_path, conf=0.5, verbose=False)

    print(f"\n{img_name}:")
    for box in results[0].boxes:
        class_name = model_v3.names[int(box.cls[0])]
        confidence = float(box.conf[0])
        print(f"  - {class_name}: {confidence:.1%}")